# OneBox vs Blackrock: how much does synchronization fix?

The same signals are recorded twice: the 14 s sync burst, eye-x, eye-y, pupil and the
photodiode go to the OneBox (XA0–XA4) **and** to Blackrock. Each system has its own
clock, so the two copies drift apart. This notebook puts both on Blackrock (NSP) time
and checks how well they agree.

- **before**: OneBox seconds (`sample / obSampRate`) plus *one* constant offset that
  lines up the start. That is what you would have without synchronization, and its
  error is the clock drift.
- **after**: `onebox_to_blackrock` from `time_map.json`, stamped into the analog export.

Both are measured on the same pairs of events. The **14 s burst pulses** were held out
of the fit, so they are the honest test. The **1 Hz** edges were used to fit the map,
so they only show how well it fits. A **photodiode** that Blackrock also records gives
a second independent check.

**Needs:** `run_exporting_pipeline.py` run with `export_analog: true`, so
`export/*.analog.mat` exists and is on Blackrock time. To overlay a channel, also fill in
the Blackrock channel ids that carry it below.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

import spikesorting as ss
from spikesorting._io import analog
from spikesorting._plots import sync as plots
from spikesorting._sync import compare, extract

# ---------------- edit these ----------------
SESSION = "session_template_1probe"          # the session config's name
MACHINE = {"darwin": "mac", "win32": "windows_rig"}.get(sys.platform, "hpc")
MINUTES = 5                                  # first / last N minutes to overlay
WORST_WINDOW_S = 1.0                         # half-width around the worst event
ZOOM_MS = 5.0                                # half-width of the zoom on that edge

# Blackrock's copy of each OneBox signal: {OneBox label: Blackrock channel id}.
# None = not recorded on Blackrock (drawn from the OneBox alone).
BLACKROCK_FILE = None                        # None -> the session's blackrock.sync_file
BLACKROCK_CHANNELS = {
    "sync_burst": "burst_channel",           # taken from the session
    "eye_x": None,
    "eye_y": None,
    "pupil": None,
    "photodiode": None,
}
# OneBox channels whose rising edges are compared as events, beside the sync burst.
EVENT_CHANNELS = ["photodiode"]

## 1. Load both systems

In [ ]:
config = ss.load_session_config(SESSION, MACHINE)
_, npx = config.per_probe()[0]              # one OneBox per run: it sits with the first probe

onebox = analog.load_onebox_analog(analog.find_onebox_analog(npx.paths.export_for("neuropixels")))
print(f"{onebox.path.name}: {', '.join(onebox.names)}")
print(f"  {onebox.fs:.4f} Hz, {onebox.n_samples / onebox.fs / 60:.1f} min")
print(f"  onebox_to_blackrock: slope {onebox.slope:.9f} ({(onebox.slope - 1) * 1e6:+.2f} ppm), "
      f"intercept {onebox.intercept:.6f} s")

channels = {
    name: (config.blackrock.burst_channel if ch == "burst_channel" else ch)
    for name, ch in BLACKROCK_CHANNELS.items() if ch is not None
}
blackrock = None
if channels:
    blackrock = analog.open_blackrock_analog(
        BLACKROCK_FILE or config.blackrock.sync_file, channels,
        gap_tolerance_ms=config.blackrock.gap_tolerance_ms,
        tolerance_s=config.alignment_tolerance_s,
    )
    print(f"{blackrock.path.name}: {channels}  |  {blackrock.time_map.summary()}")

t_start, t_stop = analog.get_recording_overlap(onebox, blackrock)
print(f"both recording for {(t_stop - t_start) / 60:.1f} min")

## 2. Events seen by both systems

The sync burst and 1 Hz edges come from the edge files `extract_sync` wrote: Blackrock's
are already on the NSP clock, the OneBox's on its own. Photodiode edges are detected
here from both traces. The threshold is set automatically from the first 5 minutes;
pass `threshold=` to override it.

In [ ]:
npx_sync, br_sync = npx.paths.sync_for("neuropixels"), config.paths.sync_for("blackrock")
events = {
    "14 s burst (held out)": (
        analog.read_edge_file(br_sync / f"{extract.BR_BURST}.txt"),
        analog.read_edge_file(npx_sync / f"{extract.NPX_BURST}.txt"),
    ),
    "1 Hz (used in fit)": (
        analog.read_edge_file(br_sync / f"{extract.BR_1HZ}.txt"),
        analog.read_edge_file(npx_sync / f"{extract.NPX_OBX_1HZ}.txt"),
    ),
}
for name in EVENT_CHANNELS:
    if blackrock is None or name not in blackrock.streams:
        print(f"{name}: not mapped to a Blackrock channel, skipped")
        continue
    obx_t, obx_thr = analog.detect_onebox_events(onebox, name)
    br_t, br_thr = analog.detect_blackrock_events(blackrock, name)
    print(f"{name}: {obx_t.size} OneBox edges (thr {obx_thr:.3g} V), "
          f"{br_t.size} Blackrock edges (thr {br_thr:.3g})")
    events[name] = (br_t, obx_t)

## 3. Quantify: the error before and after synchronization

In [ ]:
comparisons = [
    compare.compare_event_times(br, obx, onebox.slope, onebox.intercept, label=name)
    for name, (br, obx) in events.items()
]
summaries = [compare.summarize_asynchrony(c) for c in comparisons]
table = pd.DataFrame(summaries, index=[c.label for c in comparisons])
table["unpaired_blackrock"] = [c.n_blackrock - c.n_pairs for c in comparisons]
table["unpaired_onebox"] = [c.n_onebox - c.n_pairs for c in comparisons]
table.round(4)

In [ ]:
plots.plot_asynchrony_before_after(comparisons, summaries)
plt.show()

## 4. The largest unsynchronized offset, before vs after

This is where drift has pulled the two copies furthest apart, usually near the end.
On the left the OneBox trace runs late or early by that amount; on the right the map
has put it back on the Blackrock trace.

In [ ]:
ref = comparisons[0]                                  # the held-out burst
i = compare.largest_asynchrony(ref, "before")
t_event = ref.blackrock_s[i]
half = max(2 * abs(ref.before_s[i]), 0.05)
print(f"largest offset before sync: {ref.before_s[i] * 1e3:+.2f} ms "
      f"at {(t_event - t_start) / 60:.1f} min; after: {ref.after_s[i] * 1e6:+.1f} us")

before = analog.get_window_traces(onebox, blackrock, t_event - half, t_event + half,
                                  onebox_offset_s=ref.offset_s)
after = analog.get_window_traces(onebox, blackrock, t_event - half, t_event + half)
plots.plot_window_before_after(before, after, f"largest unsynchronized offset ({ref.label})",
                               marker_s=t_event, time_origin_s=t_event)
plt.show()

## 5. Synchronized signals: first and last minutes

Blackrock is solid black, OneBox dashed orange, both on Blackrock time and scaled 0–1
(the two record different units). Long windows are drawn as a min/max envelope, so
short pulses stay visible.

In [ ]:
first = analog.get_window_traces(onebox, blackrock, t_start, t_start + 60 * MINUTES)
plots.plot_window_traces(first, f"first {MINUTES} min, synchronized", time_origin_s=t_start)
plt.show()

last = analog.get_window_traces(onebox, blackrock, t_stop - 60 * MINUTES, t_stop)
plots.plot_window_traces(last, f"last {MINUTES} min, synchronized", time_origin_s=t_stop - 60 * MINUTES)
plt.show()

## 6. Where synchronization is worst

The single event with the largest error *after* the map, across every event set: first
the surrounding ±`WORST_WINDOW_S`, then a zoom on the edge itself.

In [ ]:
worst = max(comparisons, key=lambda c: np.abs(c.after_s).max())
j = compare.largest_asynchrony(worst, "after")
t_worst = worst.blackrock_s[j]
print(f"largest remaining asynchrony: {worst.after_s[j] * 1e3:+.4f} ms ({worst.label}) "
      f"at {(t_worst - t_start) / 60:.2f} min")

for half, label in ((WORST_WINDOW_S, f"+/-{WORST_WINDOW_S:g} s"), (ZOOM_MS / 1e3, f"+/-{ZOOM_MS:g} ms")):
    traces = analog.get_window_traces(onebox, blackrock, t_worst - half, t_worst + half)
    plots.plot_window_traces(traces, f"largest remaining asynchrony ({worst.label}), {label}",
                             marker_s=t_worst, time_origin_s=t_worst)
    plt.show()